# 20_02_自作の_Dataset_と_公開データセット

コード：`ch20/blocks/20_02_自作の_Dataset_と_公開データセット.ipynb`

本文のコードをコードセルで直接実行します。

モデル・データの取得、または長い計算を含みます。

上からセルを実行してください。ローカルではコード一式と同じPython環境をJupyterのカーネルに選びます。Colabでコード一式が必要な場合は、準備セルで配布ZIPをアップロードします。実行確認の範囲は `VALIDATION_COLAB_20261110.md` を参照してください。


In [ ]:
import subprocess
import sys

# 既に導入済みなら不要な更新をせず、満たしていない依存だけを導入する。
subprocess.run([sys.executable, "-m", "pip", "install", *['numpy>=1.26,<3', 'torch>=2.6,<3', 'torchvision>=0.21,<1', 'mcp==2.3.0', 'PyYAML==6.0.3', 'opentelemetry-sdk==1.45.2', 'opentelemetry-exporter-otlp-proto-http==1.45.2']], check=True)


In [ ]:
# 付録 A-2：Dataset（1 件ずつ返す）と DataLoader（まとめて・混ぜて渡す）
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

class Squares(Dataset):                                   # 自作の Dataset：x と x² の組を返す
    def __init__(self, n):
        self.x = torch.arange(n, dtype=torch.float32)
    def __len__(self):                                    # 件数
        return len(self.x)
    def __getitem__(self, i):                             # i 番目の 1 件
        return self.x[i], self.x[i] ** 2

loader = DataLoader(Squares(10), batch_size=4, shuffle=True, generator=torch.Generator().manual_seed(0))
for xb, yb in loader:
    print("バッチ", xb.tolist(), "→", yb.tolist())

# 公開データセット：FashionMNIST（28×28 の衣類の画像、10 種類）。ToTensor で 0〜1 のテンソルにする
train = datasets.FashionMNIST("data", train=True, download=True, transform=ToTensor())
test = datasets.FashionMNIST("data", train=False, download=True, transform=ToTensor())
img, label = train[0]
print("件数", len(train), len(test), " 1 件の形", tuple(img.shape), " 値の範囲", img.min().item(), "〜", img.max().item(),
      " ラベル", label, train.classes[label])
xb, yb = next(iter(DataLoader(train, batch_size=64)))
print("DataLoader の 1 バッチ", tuple(xb.shape), tuple(yb.shape))
